In [ ]:
# Cell 1: Imports and Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

# Set project root
project_root = Path().cwd().parent if Path().cwd().name == 'notebooks' else Path().cwd()
raw_path = project_root / 'data' / 'raw'

print(f"Project root: {project_root}")
print(f"Data path: {raw_path}")

# Cell 2: Load and Display File Info
files = list(raw_path.glob('*'))
print(f"Found {len(files)} files:")

for file_path in files:
    print(f"\n{'='*60}")
    print(f"📄 FILE: {file_path.name}")
    print(f"{'='*60}")
    
    # Get file info
    size_mb = file_path.stat().st_size / (1024 * 1024)
    print(f"Size: {size_mb:.2f} MB")
    print(f"Extension: {file_path.suffix}")
    
    # Try to read and display basic info
    try:
        if file_path.suffix.lower() == '.csv':
            # Try different encodings
            encodings = ['utf-8', 'latin-1', 'ISO-8859-1', 'cp1252']
            df = None
            for enc in encodings:
                try:
                    df = pd.read_csv(file_path, encoding=enc, nrows=100)
                    print(f"✓ Read with encoding: {enc}")
                    break
                except Exception as e:
                    continue
            
            if df is None:
                print("✗ Could not read with any encoding")
                continue
                
        elif file_path.suffix.lower() in ['.xlsx', '.xls']:
            # Try reading Excel
            try:
                # Try to get sheet names
                xl = pd.ExcelFile(file_path)
                print(f"Sheets: {xl.sheet_names}")
                df = xl.parse(xl.sheet_names[0], nrows=100)
                print(f"✓ Read sheet: {xl.sheet_names[0]}")
            except Exception as e:
                print(f"✗ Error reading Excel: {e}")
                continue
        else:
            print(f"✗ Unsupported file type: {file_path.suffix}")
            continue
            
        # Display basic info
        print(f"Shape: {df.shape}")
        print(f"Columns ({len(df.columns)} total):")
        
        # Show column details
        col_info = []
        for col in df.columns:
            non_null = df[col].count()
            null_pct = (1 - non_null/len(df)) * 100
            dtype = str(df[col].dtype)
            sample = str(df[col].iloc[0])[:50] if non_null > 0 else "NaN"
            col_info.append({
                'Column': col,
                'Non-Null': non_null,
                'Null %': f"{null_pct:.1f}%",
                'Dtype': dtype,
                'Sample': sample
            })
        
        col_df = pd.DataFrame(col_info)
        print(col_df.to_string(index=False))
        
        # Save a preview
        preview_path = project_root / 'data' / 'interim' / f'{file_path.stem}_preview.csv'
        df.head(20).to_csv(preview_path, index=False, encoding='utf-8')
        print(f"✓ Saved preview to: {preview_path}")
        
    except Exception as e:
        print(f"✗ Error processing file: {e}")
        import traceback
        traceback.print_exc()

Project root: /Users/raoul/club_piscine_mmm
Data path: /Users/raoul/club_piscine_mmm/data/raw
Found 7 files:

📄 FILE: CalendrierFiscal.xlsx
Size: 0.34 MB
Extension: .xlsx
Sheets: ['Feuil1', 'CalendrierFiscal', 'pivot']
✓ Read sheet: Feuil1
Shape: (100, 3)
Columns (3 total):
    Column  Non-Null Null %  Dtype Sample
Unnamed: 0         4  96.0%    str    nan
Unnamed: 1        37  63.0% object    nan
Unnamed: 2        79  21.0% object    nan
✓ Saved preview to: /Users/raoul/club_piscine_mmm/data/interim/CalendrierFiscal_preview.csv

📄 FILE: Budget 2025 - 21 août.xlsx
Size: 0.07 MB
Extension: .xlsx
Sheets: ['BUDGET 2025 - 11 déc 2024']
✓ Read sheet: BUDGET 2025 - 11 déc 2024
Shape: (78, 141)
Columns (141 total):
          Column  Non-Null Null %   Dtype Sample
      Unnamed: 0         6  92.3%     str    nan
      Unnamed: 1         0 100.0% float64    NaN
      Unnamed: 2         1  98.7% float64    nan
         VERSION        49  37.2%  object    6.0
BUDGET 2025 - V6        43  44.9%  ob

## Budget Data Coverage (Updated 2026-02-07)

This data audit notebook automatically scans and analyzes all files in the `data/raw/` directory.

**Budget Files Now Included:**
- Budget_2023_.xlsx (Sheet: 'REEL-2023 AU 30 SEPTEMBRE') - 2023 budget data through September
- Budget 2024 - REEL au 5 novembre.xlsx - 2024 budget data through November
- Budget 2025 - 21 août.xlsx - 2025 budget data

The audit will process all three years of budget data, examining:
- File structure and sheet names
- Column definitions and data types
- Missing values and data quality
- Sample data previews

All previews are saved to `data/interim/` for further review.